[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/adv-06-07-research-stopping-and-multi-agent-economics.ipynb)

# Advanced Discussion 6 and 7 — stopping rules, topologies, auctions and correlated voting

Research stopping rules, message counts by topology, task auctions with capacity caps, and majority voting with correlated errors.

**Optional advanced-discussion lab** for AI Agents and Agentic AI with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Simulation only; runs in about a minute.

In [ ]:
%pip install -q numpy

In [ ]:
import itertools, math, numpy as np

## 1. when should a research agent stop searching? (a simulation with diminishing returns)

In [ ]:
K = 8                                                                   # facts needed for a complete answer
def research(rule, param, seed, max_q=40):
    r = np.random.RandomState(seed); found = 0; q = 0; since_new = 0
    while q < max_q:
        q += 1
        p_new = 0.75 * ((K - found) / K) ** 1.5 + 0.02                # easy facts first, the last ones are hard to surface
        new = found < K and r.rand() < p_new
        found += new; since_new = 0 if new else since_new + 1
        if found >= K: break
        if rule == "budget" and q >= param: break
        if rule == "patience" and since_new >= param: break
        if rule == "marginal" and q >= 3 and p_new < param: break        # the agent's own estimate of the chance the next query helps
    return found / K, q
print("rule                        coverage of needed facts   mean queries used   coverage per query")
for name, rule, param in (("fixed budget 5", "budget", 5), ("fixed budget 10", "budget", 10), ("fixed budget 20", "budget", 20), ("stop after 2 misses in a row", "patience", 2),
                          ("stop after 3 misses in a row", "patience", 3), ("stop when P(new) < 0.15", "marginal", 0.15), ("stop when P(new) < 0.05", "marginal", 0.05)):
    res = [research(rule, param, s) for s in range(4000)]; cov = np.mean([x[0] for x in res]); qs = np.mean([x[1] for x in res])
    print("%-28s %.3f                      %5.1f              %.3f" % (name, cov, qs, cov / qs))

## 2. multi-agent: how much talking does a topology cost?

In [ ]:
print("\nmessages per round for n agents")
print("  n   orchestrator-workers (2n)   ring (n)   full mesh n(n-1)   debate, 3 rounds (3 n(n-1))")
for n in (3, 5, 8, 12): print("%3d   %14d              %6d     %10d         %12d" % (n, 2 * n, n, n * (n - 1), 3 * n * (n - 1)))

## 3. allocating tasks: auction (lowest bid wins) vs round-robin vs random

In [ ]:
rs = np.random.RandomState(0)
n_agents, n_tasks, n_types = 5, 400, 4
skill_cost = rs.uniform(1, 10, (n_agents, n_types))                      # cost for agent i to do a task of type t
types = rs.randint(0, n_types, n_tasks)
def total(assign): return sum(skill_cost[a, t] for a, t in zip(assign, types))
rr = [i % n_agents for i in range(n_tasks)]; rnd = list(rs.randint(0, n_agents, n_tasks)); auction = [int(np.argmin(skill_cost[:, t])) for t in types]
print("\ntotal cost of 400 tasks over 5 specialised agents: round-robin %.0f | random %.0f | auction (each task to the lowest bid) %.0f" % (total(rr), total(rnd), total(auction)))
load = np.bincount(auction, minlength=n_agents); print("auction load per agent:", load.tolist(), "- cheapest is not the same as balanced")
# with a load cap of 100 tasks per agent
cap = 100; load = np.zeros(n_agents, int); capped = []
for t in types:
    for a in np.argsort(skill_cost[:, t]):
        if load[a] < cap: capped.append(int(a)); load[a] += 1; break
print("auction with a cap of 100 tasks per agent: total cost %.0f, load %s" % (total(capped), load.tolist()))

## 4. do more agents help? majority vote with independent vs correlated mistakes

In [ ]:
def team_accuracy(n_agents, p, rho, trials=40000, seed=0):
    r = np.random.RandomState(seed); shared = r.rand(trials, 1); own = r.rand(trials, n_agents)
    u = np.where(r.rand(trials, n_agents) < rho, shared, own)               # with probability rho an agent copies the shared 'difficulty' draw
    correct = (u < p); return (correct.sum(1) > n_agents / 2).mean()
print("\nmajority vote of n agents, each 70% accurate, mistakes correlated by rho (share of judgement copied from a common source)")
print("  rho    n=1    n=3    n=5    n=9")
for rho in (0.0, 0.3, 0.6, 0.9): print("  %.1f   %.3f  %.3f  %.3f  %.3f" % (rho, *[team_accuracy(n, 0.70, rho) for n in (1, 3, 5, 9)]))

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.